In [1]:
import os
import numpy as np
import pandas as pd
import pyreadstat

#Esribir la direccion en donde se descargo la carpeta "Parcial"
#Ejemplo: 
# C:\Users\usuario\OneDrive\Desktop\PUCP\PYTHON\Parcial\BASE_DE_DATOS_PHYTON

RUTA = input("Pega la direccion de la carpeta BASE DE DATOS: ")
RUTA = RUTA.strip()        
RUTA = RUTA.replace('"', '')
if not os.path.exists(RUTA):
    print("OJO: esa carpeta no existe, revisa la direccion")
ANIOS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [2]:
CARPETAS = {
    "enapres":   "Victimizacion",
    "pobreza":   "Pobreza",
    "migracion": "Migracion",
    "desempleo": "Desempleo",
    "gasto":     "Gasto en seguridad",
    "canon_loc": "Canon minero/Canon local",
    "canon_gr":  "Canon minero/Canon regional",
    "vab":       "Valor Agregado Bruto",
    "ipc":       "IPC",
    "poblacion": "Poblacion",
}

In [3]:
ARCH_IPC = "n02_indice-precios_al_consumidor-nivel_nacional_set26.xlsx"
ARCH_POBLACION = "6894980-peru-poblacion-total-proyectada-al-30-de-junio-de-cada-ano-segun-departamento-provincia-y-distrito-2018-2026.xlsx"

COD_DEP = {1: 'AMAZONAS', 2: 'ANCASH', 3: 'APURIMAC', 4: 'AREQUIPA', 5: 'AYACUCHO',
           6: 'CAJAMARCA', 7: 'CALLAO', 8: 'CUSCO', 9: 'HUANCAVELICA', 10: 'HUANUCO',
           11: 'ICA', 12: 'JUNIN', 13: 'LA LIBERTAD', 14: 'LAMBAYEQUE', 15: 'LIMA',
           16: 'LORETO', 17: 'MADRE DE DIOS', 18: 'MOQUEGUA', 19: 'PASCO', 20: 'PIURA',
           21: 'PUNO', 22: 'SAN MARTIN', 23: 'TACNA', 24: 'TUMBES', 25: 'UCAYALI'}

DEPTOS = list(COD_DEP.values())

REGION_NATURAL = {
    "TUMBES": "Costa", "PIURA": "Costa", "LAMBAYEQUE": "Costa", "LA LIBERTAD": "Costa",
    "ANCASH": "Costa", "LIMA": "Costa", "CALLAO": "Costa", "ICA": "Costa",
    "AREQUIPA": "Costa", "MOQUEGUA": "Costa", "TACNA": "Costa",
    "CAJAMARCA": "Sierra", "HUANUCO": "Sierra", "PASCO": "Sierra", "JUNIN": "Sierra",
    "HUANCAVELICA": "Sierra", "AYACUCHO": "Sierra", "APURIMAC": "Sierra",
    "CUSCO": "Sierra", "PUNO": "Sierra",
    "LORETO": "Selva", "UCAYALI": "Selva", "MADRE DE DIOS": "Selva",
    "SAN MARTIN": "Selva", "AMAZONAS": "Selva"}

# Victimizacion (factor de expansion)
CONFIG_VICTIM = {
    2018: ("P601", "FACTOR"),
    2019: ("P601", "FACTOR"),
    2020: ("P601", "FACTOR_CALIBRADO"),
    2021: ("P601", "FACTOR_CALIBRADO"),
    2022: ("P601", "FACTOR"),
    2023: ("P601", "FACTOR"),
    2024: ("P615", "FACTOR_CAP600B"),  
    2025: ("P424", "FACTOR_CAP400"),
}

# Orden final de las columnas
COLUMNAS_FINALES = [
    "depto", "victim_100k", "n_victim", "anio", "pobreza", "pobreza_extrema", "n_hogares",
    "migracion", "n_personas", "desempleo", "n_pea", "poblacion", "crec_pob", "gasto_seg",
    "canon_gr", "canon_loc", "vab_total", "vab_min", "vab_sin_min", "deflactor",
    "canon_total", "canon_pc_real", "canon_pc_real_l1", "ln_canon_l1",
    "gasto_seg_pc_real", "pbi_sinmin_pc", "ln_pbi_sinmin_pc", "prod_min_pc",
    "region_natural"]

In [4]:
# ============================================================
# 1. FUNCIONES PARA LEER ARCHIVOS Y LIMPIAR NOMBRES
# ============================================================
def armar_ruta(clave, nombre_archivo):
    carpeta = CARPETAS[clave]
    if carpeta == "":
        ruta = RUTA + "/" + nombre_archivo
    else:
        ruta = RUTA + "/" + carpeta + "/" + nombre_archivo
    return ruta


def leer_base(ruta, columnas=None, hoja=0, cabecera=0, tipos=None, html=False):
    # Separo el nombre del archivo por los puntos y nos quedamos con lo ultimo
    partes = ruta.split(".")
    extension = partes[-1].lower()

    if extension == "csv":
        df = pd.read_csv(ruta, encoding="latin1", sep=None, engine="python")
    elif extension == "dta":
        if columnas is None:
            df = pd.read_stata(ruta, convert_categoricals=False)
        else:
            df = pd.read_stata(ruta, convert_categoricals=False, columns=columnas)
    elif extension == "sav":
        df, meta = pyreadstat.read_sav(ruta)
    elif extension == "xls" or extension == "xlsx":
        if html == True:
            df = pd.read_html(ruta, encoding="latin1", decimal=".", thousands=",")
        else:
            df = pd.read_excel(ruta, sheet_name=hoja, header=cabecera, dtype=tipos)
    else:
        print("Extension no reconocida:", extension)
        df = None

    return df

In [5]:
def limpiar_texto(serie):
    serie = serie.astype(str).str.upper().str.strip()
    cambios = [("Á", "A"), ("É", "E"), ("Í", "I"), ("Ó", "O"), ("Ú", "U"), ("Ü", "U"), ("Ñ", "N")]
    for con_tilde, sin_tilde in cambios:
        serie = serie.str.replace(con_tilde, sin_tilde, regex=False)
    return serie


def limpiar_depto(serie):
    serie = limpiar_texto(serie)
    serie = serie.replace({"PROV. CONST. DEL CALLAO": "CALLAO",
                           "PROVINCIA CONSTITUCIONAL DEL CALLAO": "CALLAO",
                           "REGION LIMA": "LIMA",
                           "PROVINCIA DE LIMA": "LIMA",
                           "LIMA PROVINCIAS": "LIMA",
                           "LIMA METROPOLITANA": "LIMA"})
    return serie

In [6]:
# ============================================================
# 2. UNA FUNCION PARA CADA VARIABLE
# ============================================================

In [7]:
# ------------------------------------------------------------
# 2.1 VICTIMIZACION 
# ------------------------------------------------------------
def tasa_victim(df, anio, prefijo, factor):
    df.columns = df.columns.str.upper()

    items = []
    for c in df.columns:
        if c.startswith(prefijo + "_"):
            items.append(c)

    for c in items:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[factor] = df[factor].astype(str).str.replace(",", ".")
    df[factor] = pd.to_numeric(df[factor], errors="coerce")
    df = df.copy()
    df["depto"] = limpiar_depto(df["NOMBREDD"])

    df["victima"] = (df[items] == 1).any(axis=1)
    df["respondio"] = df[items].isin([1, 2, 3]).any(axis=1)
    df["peso_victima"] = df["victima"] * df[factor]

    df = df[df["respondio"] & df[factor].notna()]

    g = df.groupby("depto")[["peso_victima", factor]].sum()
    tasa = g["peso_victima"] / g[factor] * 100000
    n = df.groupby("depto").size()

    salida = pd.DataFrame({"victim_100k": tasa, "n_victim": n}).reset_index()
    salida["anio"] = anio
    return salida


def calc_victimizacion():
    resultados = []
    for anio in ANIOS:
        # 2018 y 2019 son .sav, los demas son .csv
        if anio == 2018 or anio == 2019:
            archivo = "ENAPRES_" + str(anio) + ".sav"
        else:
            archivo = "ENAPRES_" + str(anio) + ".csv"

        df = leer_base(armar_ruta("enapres", archivo))
        prefijo = CONFIG_VICTIM[anio][0]
        factor = CONFIG_VICTIM[anio][1]
        resultados.append(tasa_victim(df, anio, prefijo, factor))

    panel = pd.concat(resultados, ignore_index=True)
    return panel[["depto", "victim_100k", "n_victim", "anio"]]

In [8]:
# ------------------------------------------------------------
# 2.2 POBREZA 
# ------------------------------------------------------------
def pobreza_un_anio(anio):
    df = leer_base(armar_ruta("pobreza", "pob_" + str(anio) + ".dta"))
    df.columns = df.columns.str.lower()

    df["pobreza"] = pd.to_numeric(df["pobreza"], errors="coerce")
    df["mieperho"] = pd.to_numeric(df["mieperho"], errors="coerce")
    df["factor07"] = pd.to_numeric(df["factor07"], errors="coerce")

    df["ubigeo"] = df["ubigeo"].astype(str).str.zfill(6)
    df["depto"] = df["ubigeo"].str[:2].astype(int).map(COD_DEP)

    df["facpob"] = df["factor07"] * df["mieperho"]
    df = df.dropna(subset=["pobreza", "facpob", "depto"]).copy()

    df["pobre"] = df["pobreza"].isin([1, 2]).astype(int)    
    df["extremo"] = (df["pobreza"] == 1).astype(int)        
    df["w_pobre"] = df["pobre"] * df["facpob"]
    df["w_extremo"] = df["extremo"] * df["facpob"]

    g = df.groupby("depto").agg(w_pobre=("w_pobre", "sum"),
                                w_extremo=("w_extremo", "sum"),
                                pob=("facpob", "sum"),
                                n_hogares=("pobre", "size")).reset_index()

    g["pobreza"] = g["w_pobre"] / g["pob"] * 100
    g["pobreza_extrema"] = g["w_extremo"] / g["pob"] * 100
    g["anio"] = anio
    return g[["depto", "anio", "pobreza", "pobreza_extrema", "n_hogares"]]


def calc_pobreza():
    resultados = []
    for anio in ANIOS:
        resultados.append(pobreza_un_anio(anio))
    return pd.concat(resultados, ignore_index=True)

In [9]:
# ------------------------------------------------------------
# 2.3 MIGRACION 
# ------------------------------------------------------------
def migracion_un_anio(anio):
    df = leer_base(armar_ruta("migracion", "migra_" + str(anio) + ".dta"),
                   columnas=["ubigeo", "p401f", "p401g", "factor07"])
    df.columns = df.columns.str.lower()

    df["ubigeo"] = df["ubigeo"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
    df["depto"] = df["ubigeo"].str[:2].astype(int).map(COD_DEP)

    df["p401f"] = pd.to_numeric(df["p401f"], errors="coerce")
    df["p401g"] = pd.to_numeric(df["p401g"], errors="coerce")
    df["factor07"] = pd.to_numeric(df["factor07"], errors="coerce")

    # Departamento donde vivia hace 5 años
    df["depto_prev"] = df["p401g"]
    filtro = df["p401g"] > 100
    df.loc[filtro, "depto_prev"] = df.loc[filtro, "p401g"] // 10000
    df["depto_prev"] = df["depto_prev"].map(COD_DEP)

    df = df[df["p401f"].isin([1, 2]) & df["depto"].notna() & df["factor07"].notna()].copy()

    df["migrante"] = ((df["p401f"] == 2)
                      & df["depto_prev"].notna()
                      & (df["depto_prev"] != df["depto"])).astype(int)

    df["w_mig"] = df["migrante"] * df["factor07"]

    g = df.groupby("depto").agg(w_mig=("w_mig", "sum"),
                                pob=("factor07", "sum"),
                                n_personas=("migrante", "size")).reset_index()
    g["migracion"] = g["w_mig"] / g["pob"] * 100
    g["anio"] = anio

    return g[["depto", "anio", "migracion", "n_personas"]]


def calc_migracion():
    resultados = []
    for anio in ANIOS:
        resultados.append(migracion_un_anio(anio))
    return pd.concat(resultados, ignore_index=True)

In [10]:
# ------------------------------------------------------------
# 2.4 DESEMPLEO 
# ------------------------------------------------------------
def desempleo_un_anio(anio):
    df = leer_base(armar_ruta("desempleo", "desempleo_" + str(anio) + ".dta"))
    df.columns = df.columns.str.upper()

    if "FAC500A" in df.columns:
        factor = "FAC500A"
    elif "FAC500" in df.columns:
        factor = "FAC500"
    else:
        raise KeyError(str(anio) + ": no encontre el factor de expansion")

    df = df[["UBIGEO", "OCU500", factor]].copy()
    df["OCU500"] = pd.to_numeric(df["OCU500"], errors="coerce")
    df[factor] = pd.to_numeric(df[factor], errors="coerce")
    df["UBIGEO"] = df["UBIGEO"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
    df["cod_dpto"] = df["UBIGEO"].str[:2].astype(int)

    # PEA = ocupados (1) + desocupados abiertos (2) + desocupados ocultos (3)
    pea = df[df["OCU500"].isin([1, 2, 3]) & df[factor].notna()].copy()
    pea["des_pond"] = pea["OCU500"].isin([2, 3]) * pea[factor]

    t = pea.groupby("cod_dpto").agg(des_pond=("des_pond", "sum"),
                                    pea_pond=(factor, "sum"),
                                    n_pea=("OCU500", "size")).reset_index()
    t["desempleo"] = t["des_pond"] / t["pea_pond"] * 100
    t["anio"] = anio
    return t[["cod_dpto", "anio", "desempleo", "n_pea"]]


def calc_desempleo():
    resultados = []
    for anio in ANIOS:
        resultados.append(desempleo_un_anio(anio))
    d = pd.concat(resultados, ignore_index=True)
    d["depto"] = d["cod_dpto"].map(COD_DEP)
    return d[["depto", "anio", "desempleo", "n_pea"]]

In [11]:
# ------------------------------------------------------------
# 2.5 POBLACION 
# ------------------------------------------------------------
def calc_poblacion():
    # El archivo trae titulos arriba, por eso no uso encabezado 
    raw = leer_base(armar_ruta("poblacion", ARCH_POBLACION), cabecera=None, tipos={0: str})
    raw = raw.iloc[:, :11]
    raw.columns = ["ubigeo", "nombre", 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
    raw["ubigeo"] = raw["ubigeo"].astype(str).str.strip()

    dep = raw[raw["ubigeo"].str.match(r"^\d{2}0000$") & (raw["ubigeo"] != "000000")].copy()
    dep["depto"] = dep["ubigeo"].str[:2].astype(int).map(COD_DEP)

    p = dep.melt(id_vars="depto", value_vars=[2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026],
                 var_name="anio", value_name="poblacion")
    p["anio"] = p["anio"].astype(int)
    p["poblacion"] = p["poblacion"].astype(float)
    p = p.sort_values(["depto", "anio"]).reset_index(drop=True)

    p["crec_pob"] = p.groupby("depto")["poblacion"].pct_change() * 100

    p = p[p["anio"] <= 2025]
    return p[["depto", "anio", "poblacion", "crec_pob"]]

In [12]:
# ------------------------------------------------------------
# 2.6 GASTO EN SEGURIDAD 
# ------------------------------------------------------------
def gasto_un_anio(anio):
    tablas = leer_base(armar_ruta("gasto", "gasto_seg_" + str(anio) + ".xls"), html=True)
    d = tablas[-1].iloc[:, [0, 6]].copy()     
    d.columns = ["depto_raw", "gasto_seg"]
    d["depto"] = d["depto_raw"].astype(str).str[:2].astype(int).map(COD_DEP)
    d["gasto_seg"] = pd.to_numeric(d["gasto_seg"], errors="coerce")
    d["anio"] = anio
    return d[["depto", "anio", "gasto_seg"]]


def calc_gasto():
    resultados = []
    for anio in ANIOS:
        resultados.append(gasto_un_anio(anio))
    g = pd.concat(resultados, ignore_index=True)
    return g.sort_values(["depto", "anio"]).reset_index(drop=True)


In [13]:
# ------------------------------------------------------------
# 2.7 CANON MINERO REGIONAL 
# ------------------------------------------------------------
def canon_regional_un_anio(anio):
    t = leer_base(armar_ruta("canon_gr", "canon_region_" + str(anio) + ".xls"), html=True)
    df = t[3].copy()
    df.columns = ["pliego", "PIA", "PIM", "canon_gr"]
    df["pliego"] = df["pliego"].astype(str)

    df["depto"] = df["pliego"].str.extract(r"DEPARTAMENTO DE (.+)$")[0]
    df.loc[df["pliego"].str.contains("CALLAO"), "depto"] = "CALLAO"
    df = df.dropna(subset=["depto"]).copy()    

    df["canon_gr"] = pd.to_numeric(df["canon_gr"], errors="coerce")
    df["anio"] = anio
    return df[["depto", "anio", "canon_gr"]]


def calc_canon_regional():
    resultados = []
    for anio in ANIOS:
        resultados.append(canon_regional_un_anio(anio))
    canon = pd.concat(resultados, ignore_index=True)

    filas = []
    for depto in DEPTOS:
        for anio in ANIOS:
            filas.append({"depto": depto, "anio": anio})
    base = pd.DataFrame(filas)

    canon = base.merge(canon, on=["depto", "anio"], how="left")
    canon["canon_gr"] = canon["canon_gr"].fillna(0)

    return canon

In [14]:
# ------------------------------------------------------------
# 2.8 CANON MINERO LOCAL 
# ------------------------------------------------------------
def canon_local_un_anio(anio):
    tablas = leer_base(armar_ruta("canon_loc", "canon_local_" + str(anio) + ".xls"), html=True)
    d = tablas[-1].iloc[:, [0, 3]].copy()
    d.columns = ["depto_raw", "canon_loc"]
    d["cod_dpto"] = d["depto_raw"].astype(str).str[:2].astype(int)
    d["canon_loc"] = pd.to_numeric(d["canon_loc"], errors="coerce")

    filas = []
    for cod in range(1, 26):
        valor = d.loc[d["cod_dpto"] == cod, "canon_loc"].sum()
        filas.append({"cod_dpto": cod, "anio": anio, "canon_loc": valor})
    return pd.DataFrame(filas)


def calc_canon_local():
    resultados = []
    for anio in ANIOS:
        resultados.append(canon_local_un_anio(anio))
    c = pd.concat(resultados, ignore_index=True)
    c["depto"] = c["cod_dpto"].map(COD_DEP)
    return c[["depto", "anio", "canon_loc"]]

In [15]:
# ------------------------------------------------------------
# 2.9 VALOR AGREGADO BRUTO 
# ------------------------------------------------------------
def vab_un_archivo(ruta, nombre):
    d = leer_base(ruta, hoja="cuadro1", cabecera=6)
    d = d.rename(columns={"Departamentos": "depto"}).dropna(subset=["depto"])

    nuevas = []
    for c in d.columns:
        if c == "depto":
            nuevas.append(c)
        else:
            nuevas.append(str(c)[:4])
    d.columns = nuevas

    d["depto"] = limpiar_texto(d["depto"])
    d = d[d["depto"] != "LIMA"]  
    d["depto"] = d["depto"].replace({"PROV. CONST. DEL CALLAO": "CALLAO", "REGION LIMA": "LIMA",
                                     "PROVINCIA DE LIMA": "LIMA", "LIMA PROVINCIAS": "LIMA",
                                     "LIMA METROPOLITANA": "LIMA"})
    d = d[d["depto"].isin(DEPTOS)]

    d = d.melt(id_vars="depto", var_name="anio", value_name=nombre)
    d["anio"] = d["anio"].astype(int)
    d[nombre] = pd.to_numeric(d[nombre], errors="coerce")
    d = d[(d["anio"] >= 2018) & (d["anio"] <= 2025)]

    return d.groupby(["depto", "anio"], as_index=False)[nombre].sum()


def calc_vab():
    vab_total = vab_un_archivo(armar_ruta("vab", "pbi_peru_17.xlsx"), "vab_total")
    vab_min = vab_un_archivo(armar_ruta("vab", "pbi_act03_18.xlsx"), "vab_min")

    vab = vab_total.merge(vab_min, on=["depto", "anio"], validate="one_to_one")
    vab["vab_sin_min"] = vab["vab_total"] - vab["vab_min"]
    return vab

In [16]:
# ------------------------------------------------------------
# 2.10 IPC y DEFLACTOR 
# ------------------------------------------------------------
def calc_ipc():
    ipc = leer_base(armar_ruta("ipc", ARCH_IPC), hoja="Base Dic.2021", cabecera=3)

    ipc["Año"] = ipc["Año"].ffill()        
    ipc = ipc.dropna(subset=["Mes"])
    ipc["Año"] = pd.to_numeric(ipc["Año"], errors="coerce")
    ipc["Índice"] = pd.to_numeric(ipc["Índice"], errors="coerce")

    ipc_anual = ipc.groupby("Año").agg(ipc=("Índice", "mean"),
                                       meses=("Índice", "count")).reset_index()
    ipc_anual = ipc_anual.rename(columns={"Año": "anio"})
    ipc_anual = ipc_anual[(ipc_anual["anio"] >= 2018) & (ipc_anual["anio"] <= 2025)]
    ipc_anual["anio"] = ipc_anual["anio"].astype(int)

    base = ipc_anual.loc[ipc_anual["anio"] == 2025, "ipc"].iloc[0]
    ipc_anual["deflactor"] = ipc_anual["ipc"] / base
    return ipc_anual[["anio", "deflactor"]]

In [17]:
# ============================================================
# 3. FUNCION QUE UNE TODO
# ============================================================
def armar_panel():
    # Calculo cada variable con su funcion
    print("--- victimizacion ---")
    victim = calc_victimizacion()
    print("--- pobreza ---")
    pobreza = calc_pobreza()
    print("--- migracion ---")
    migracion = calc_migracion()
    print("--- desempleo ---")
    desempleo = calc_desempleo()
    print("--- poblacion ---")
    poblacion = calc_poblacion()
    print("--- gasto en seguridad ---")
    gasto = calc_gasto()
    print("--- canon regional ---")
    canon_gr = calc_canon_regional()
    print("--- canon local ---")
    canon_loc = calc_canon_local()
    print("--- valor agregado bruto ---")
    vab = calc_vab()
    print("--- ipc ---")
    ipc = calc_ipc()

    # Uno todas las tablas por departamento y año
    panel = victim
    for tabla in [pobreza, migracion, desempleo, poblacion, gasto, canon_gr, canon_loc, vab]:
        panel = panel.merge(tabla, on=["depto", "anio"], how="outer", validate="one_to_one")

    panel = panel.merge(ipc, on="anio", how="left")
    panel = panel.sort_values(["depto", "anio"]).reset_index(drop=True)

    # Canon total y canon per capita en soles reales
    panel["canon_total"] = panel["canon_gr"] + panel["canon_loc"]
    panel["canon_pc_real"] = panel["canon_total"] / panel["deflactor"] / panel["poblacion"]

    # Rezago de un año y logaritmo (log1p porque hay departamentos con canon cero)
    panel["canon_pc_real_l1"] = panel.groupby("depto")["canon_pc_real"].shift(1)
    panel["ln_canon_l1"] = np.log1p(panel["canon_pc_real_l1"])

    # Gasto en seguridad per capita real
    panel["gasto_seg_pc_real"] = panel["gasto_seg"] / panel["deflactor"] / panel["poblacion"]

    # PBI sin mineria y produccion minera per capita
    panel["pbi_sinmin_pc"] = panel["vab_sin_min"] * 1000 / panel["poblacion"]
    panel["ln_pbi_sinmin_pc"] = np.log(panel["pbi_sinmin_pc"])
    panel["prod_min_pc"] = panel["vab_min"] * 1000 / panel["poblacion"]

    # Region natural
    panel["region_natural"] = panel["depto"].map(REGION_NATURAL)

    return panel[COLUMNAS_FINALES]

In [18]:
panel = armar_panel()
print("Panel final:", panel.shape)
panel.to_csv("base_final.csv", index=False)

--- victimizacion ---
--- pobreza ---
--- migracion ---
--- desempleo ---
--- poblacion ---
--- gasto en seguridad ---
--- canon regional ---
--- canon local ---
--- valor agregado bruto ---
--- ipc ---
Panel final: (200, 29)


In [19]:
# Verificamos el formato
print(panel.dtypes)

depto                    str
victim_100k          float64
n_victim               int64
anio                   int64
pobreza              float64
pobreza_extrema      float64
n_hogares              int64
migracion            float64
n_personas             int64
desempleo            float64
n_pea                  int64
poblacion            float64
crec_pob             float64
gasto_seg              int64
canon_gr             float64
canon_loc              int64
vab_total            float64
vab_min              float64
vab_sin_min          float64
deflactor            float64
canon_total          float64
canon_pc_real        float64
canon_pc_real_l1     float64
ln_canon_l1          float64
gasto_seg_pc_real    float64
pbi_sinmin_pc        float64
ln_pbi_sinmin_pc     float64
prod_min_pc          float64
region_natural           str
dtype: object


In [20]:
# Promedio 2018-2025 por departamento
resumen_depto = panel.groupby("depto")[["victim_100k", "pobreza", "desempleo", "migracion"]].mean()
print(resumen_depto.round(1))

               victim_100k  pobreza  desempleo  migracion
depto                                                    
AMAZONAS           15759.2     28.8        3.3        4.4
ANCASH             17809.1     22.7        5.1        1.8
APURIMAC           22792.4     27.5        2.9        3.4
AREQUIPA           26583.0     12.7        7.9        2.9
AYACUCHO           24420.8     38.2        4.2        2.8
CAJAMARCA          13566.7     42.1        4.9        1.4
CALLAO             26019.4     28.1        9.6        4.5
CUSCO              32116.1     22.6        3.8        1.9
HUANCAVELICA       24013.8     38.6        3.3        2.4
HUANUCO            14616.1     36.7        4.9        3.5
ICA                18060.8      5.4        4.7        2.7
JUNIN              31618.1     22.2        4.0        3.2
LA LIBERTAD        23784.9     27.5        6.8        1.8
LAMBAYEQUE         14670.3     14.1        5.0        2.6
LIMA               28409.5     23.3        8.5        2.2
LORETO        

In [21]:
# Promedio por region natural (Costa, Sierra, Selva)
resumen_region = panel.groupby("region_natural")[["victim_100k", "pobreza", "desempleo", "migracion"]].mean()
print(resumen_region.round(1))

                victim_100k  pobreza  desempleo  migracion
region_natural                                            
Costa               21660.7     19.8        6.6        2.5
Selva               19732.6     23.9        3.9        5.0
Sierra              23830.4     34.0        4.6        2.7


In [22]:
# Promedio nacional por año
resumen_anio = panel.groupby("anio")[["victim_100k", "pobreza", "desempleo", "migracion"]].mean()
print(resumen_anio.round(1))

      victim_100k  pobreza  desempleo  migracion
anio                                            
2018      24420.4     22.2        4.2        2.8
2019      24060.3     21.6        4.2        2.7
2020      20765.6     30.7        8.4        2.4
2021      15687.5     25.4        4.9        3.6
2022      20215.3     27.0        4.6        3.5
2023      23052.5     28.5        5.6        3.5
2024      25645.7     26.5        5.7        3.4
2025      22602.1     24.1        5.0        2.9
